In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = (SparkSession.builder
         .master("spark://spark-master:7077")
        .getOrCreate()
        )

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/04 01:57:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df = spark.read.parquet("/data/output/*.parquet")
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [5]:
df2 = df.select("VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count", "total_amount")
df2.show(5)

+--------+--------------------+---------------------+---------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|total_amount|
+--------+--------------------+---------------------+---------------+------------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|              1|        9.95|
|       1| 2019-01-01 00:59:47|  2019-01-01 01:18:59|              1|        16.3|
|       2| 2018-12-21 13:48:30|  2018-12-21 13:52:40|              3|         5.8|
|       2| 2018-11-28 15:52:25|  2018-11-28 15:55:45|              5|        7.55|
|       2| 2018-11-28 15:56:57|  2018-11-28 15:58:33|              5|       55.55|
+--------+--------------------+---------------------+---------------+------------+
only showing top 5 rows



In [8]:
from pyspark.sql.functions import col
df3 = df.select(
    col("VendorID"), 
    col("tpep_pickup_datetime"), 
    col("tpep_dropoff_datetime"), 
    col("passenger_count"), 
    col("total_amount")
)
df3.show(5)

+--------+--------------------+---------------------+---------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|total_amount|
+--------+--------------------+---------------------+---------------+------------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|              1|        9.95|
|       1| 2019-01-01 00:59:47|  2019-01-01 01:18:59|              1|        16.3|
|       2| 2018-12-21 13:48:30|  2018-12-21 13:52:40|              3|         5.8|
|       2| 2018-11-28 15:52:25|  2018-11-28 15:55:45|              5|        7.55|
|       2| 2018-11-28 15:56:57|  2018-11-28 15:58:33|              5|       55.55|
+--------+--------------------+---------------------+---------------+------------+
only showing top 5 rows



In [9]:
# rename columns
df4 = (df.withColumnRenamed("VendorId", "vendor_id")
        .withColumnRenamed("RatecodeID", "rate_code_id")
        .withColumnRenamed("PULocationID", "pu_location_id")
        .withColumnRenamed("DOLocationID", "do_location_id")
      )

df4.show()

+---------+--------------------+---------------------+---------------+-------------+------------+------------------+--------------+--------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|vendor_id|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|rate_code_id|store_and_fwd_flag|pu_location_id|do_location_id|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|
+---------+--------------------+---------------------+---------------+-------------+------------+------------------+--------------+--------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+
|        1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|              1|          1.5|           1|                 N|           151|           239|           1|        7.0|  0.5|    0.5|      1.65|      

In [13]:
# filtering

df_filter = df4.filter(col("passenger_count") > 2)
df_filter = df_filter.select("vendor_id", "passenger_count", "total_amount")
df_filter.show(5)

[Stage 10:>                                                         (0 + 1) / 1]

+---------+---------------+------------+
|vendor_id|passenger_count|total_amount|
+---------+---------------+------------+
|        2|              3|         5.8|
|        2|              5|        7.55|
|        2|              5|       55.55|
|        2|              5|       13.31|
|        2|              5|       55.55|
+---------+---------------+------------+
only showing top 5 rows



In [14]:
df_filter2 = df_filter.filter("passenger_count > 2")
df_filter2.show()

+---------+---------------+------------+
|vendor_id|passenger_count|total_amount|
+---------+---------------+------------+
|        2|              3|         5.8|
|        2|              5|        7.55|
|        2|              5|       55.55|
|        2|              5|       13.31|
|        2|              5|       55.55|
|        1|              3|        8.15|
|        1|              3|         7.8|
|        1|              4|         9.3|
|        1|              3|        10.3|
|        2|              3|        29.8|
|        1|              4|        57.8|
|        1|              4|       20.75|
|        1|              3|        8.15|
|        1|              4|       16.56|
|        2|              4|       19.56|
|        2|              4|         6.8|
|        2|              3|         6.8|
|        2|              3|       11.76|
|        2|              3|       14.76|
|        2|              3|        8.58|
+---------+---------------+------------+
only showing top

In [16]:
# logical operators
df_filter5 = df4.filter(
    (col("passenger_count") > 2) &
    (col("total_amount") > 10)
)

df_filter5 = df_filter5.select("vendor_id", "passenger_count", "total_amount")
df_filter5.show()

+---------+---------------+------------+
|vendor_id|passenger_count|total_amount|
+---------+---------------+------------+
|        2|              5|       55.55|
|        2|              5|       13.31|
|        2|              5|       55.55|
|        1|              3|        10.3|
|        2|              3|        29.8|
|        1|              4|        57.8|
|        1|              4|       20.75|
|        1|              4|       16.56|
|        2|              4|       19.56|
|        2|              3|       11.76|
|        2|              3|       14.76|
|        2|              3|        16.8|
|        2|              5|       24.12|
|        2|              5|        14.0|
|        1|              3|       12.35|
|        1|              3|       20.15|
|        1|              3|       10.75|
|        2|              5|       70.27|
|        1|              4|        19.8|
|        1|              4|       30.35|
+---------+---------------+------------+
only showing top